# Poisson and Matérn type-II point processes: spectral scale selection

This notebook simulates two planar point clouds in the unit square:

1. a homogeneous Poisson point process (PPP), and
2. a Matérn type-II hard-core process obtained by marked thinning of a PPP.

For each point cloud $P$, it evaluates the ball/Vietoris–Rips graph on a finite grid
$$0<\epsilon_1<\cdots<\epsilon_N<\tfrac12\operatorname{diam}(P),$$
where two points are adjacent when their distance is at most $2\epsilon$. At every scale,
$$f_r(K_\epsilon)=r\log|C(K_\epsilon)|+\log|E(K_\epsilon)|,$$
with $|C|$ the number of connected components and $|E|$ the number of edges. The scale-similarity matrix of the filtration is
$$\Sigma_{ij}=f(K_{\epsilon_{\min(i,j)}}).$$
with $r=1$ by default. Because the smallest graphs may have no edges, the implementation uses the finite convention $\log|E|:=\log(\max\{|E|,1\})$. Thus $f_r\geq0$ and no additive shift is needed. The selected scale is the earliest grid point whose coordinate in the Perron (principal) eigenvector of $\Sigma$ is maximal. A tiny positive regularization avoids numerical reducibility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import cKDTree
from scipy.cluster.hierarchy import linkage

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 2026
WINDOW = np.array([[0.0, 1.0], [0.0, 1.0]])
PPP_INTENSITY = 300.0
HARDCORE_RADIUS = 0.025
HARDCORE_DISTANCE = 2.0 * HARDCORE_RADIUS
N_EPSILONS = 120
R_WEIGHT = 1.0


## Simulation

For the Matérn-II process, every parent PPP point receives an independent uniform mark. A point survives if its mark is strictly the smallest among all parent points within the hard-core distance $h$. The parent process is simulated on an $h$-padded window and cropped after thinning; this border correction lets outside competitors remove points near the visible boundary. Consequently, retained points in the observation window are separated by at least $h$.

In [ ]:
def simulate_ppp(intensity, window, rng):
    lengths = window[:, 1] - window[:, 0]
    n = rng.poisson(intensity * np.prod(lengths))
    return rng.uniform(window[:, 0], window[:, 1], size=(n, 2))


def simulate_matern_type_ii(parent_intensity, hardcore_distance, window, rng):
    # Simulate possible competitors beyond the observation boundary.
    padded = window + np.array([[-hardcore_distance, hardcore_distance]] * 2)
    parents = simulate_ppp(parent_intensity, padded, rng)
    marks = rng.random(len(parents))
    neighborhoods = cKDTree(parents).query_ball_point(parents, hardcore_distance)
    keep = np.array([marks[i] == marks[neighbors].min() for i, neighbors in enumerate(neighborhoods)])
    retained = parents[keep]
    inside = np.all((retained >= window[:, 0]) & (retained <= window[:, 1]), axis=1)
    return retained[inside], parents


rng = np.random.default_rng(SEED)
poisson_points = simulate_ppp(PPP_INTENSITY, WINDOW, rng)
matern2_points, matern2_parents = simulate_matern_type_ii(
    PPP_INTENSITY, HARDCORE_DISTANCE, WINDOW, rng
)

point_clouds = {
    "Poisson point process": poisson_points,
    "Matérn type-II": matern2_points,
}

for name, points in point_clouds.items():
    nearest_pair = pdist(points).min() if len(points) > 1 else np.nan
    print(f"{name:24s}: n={len(points):3d}, minimum pair distance={nearest_pair:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, (name, points) in zip(axes, point_clouds.items()):
    ax.scatter(points[:, 0], points[:, 1], s=18, alpha=0.8, edgecolor="none")
    ax.set(title=f"{name} (n={len(points)})", xlim=(0, 1), ylim=(0, 1), xlabel="x", ylabel="y")
    ax.set_aspect("equal")
plt.show()

## Spectral scale-selection pipeline

The grid uses $N$ equally spaced interior points of $(0,\operatorname{diam}(P)/2)$, so neither endpoint is included. Component counts are computed by a union–find sweep over edges ordered by length; this is equivalent to rebuilding the threshold graph at each scale but is faster.

In [ ]:
def graph_counts_on_grid(points, epsilons):
    n = len(points)
    distances = pdist(points)
    thresholds = 2.0 * np.asarray(epsilons)

    # Edge counts are cumulative counts in the sorted pair-distance array.
    sorted_distances = np.sort(distances)
    edge_counts = np.searchsorted(sorted_distances, thresholds, side="right")

    # In a threshold graph, each accepted single-linkage merge reduces |C| by one.
    # The linkage merge distances are nondecreasing and encode an MST sweep.
    merge_distances = linkage(distances, method="single")[:, 2]
    accepted_merges = np.searchsorted(merge_distances, thresholds, side="right")
    component_counts = n - accepted_merges
    return component_counts, edge_counts


def principal_meet_vector(c, regularization=1e-12, tolerance=1e-11, max_iter=5000):
    """Perron vector of Sigma_ij=c_min(i,j), using O(N) matrix products."""
    c = np.asarray(c, dtype=float)
    c = c / c.max()
    vector = np.full(len(c), 1.0 / len(c))
    for _ in range(max_iter):
        prefix = np.cumsum(c * vector)
        suffix_after = np.cumsum(vector[::-1])[::-1] - vector
        updated = prefix + c * suffix_after + regularization * vector.sum()
        updated /= updated.sum()
        if np.max(np.abs(updated - vector)) < tolerance:
            return updated
        vector = updated
    raise RuntimeError("Perron iteration did not converge.")


def spectral_scale_selection(
    points, n_epsilons=120, r=1.0, regularization=1e-12
):
    diameter = pdist(points).max()
    # N points strictly inside (0, diameter/2).
    epsilons = np.linspace(0.0, 0.5 * diameter, n_epsilons + 2)[1:-1]
    components, edges = graph_counts_on_grid(points, epsilons)
    functional = r * np.log(components) + np.log(np.maximum(edges, 1))

    # Sigma_ij = f(K_{epsilon_min(i,j)}); scaling preserves eigenvectors.
    if functional.max() == 0:
        raise ValueError("The functional is constant on the epsilon grid.")
    principal = principal_meet_vector(functional, regularization=regularization)
    maximizers = np.flatnonzero(np.isclose(
        principal, principal.max(), rtol=1e-9, atol=1e-12
    ))
    selected_index = int(maximizers[0])

    return {
        "diameter": diameter,
        "epsilons": epsilons,
        "components": components,
        "edges": edges,
        "functional": functional,
        "r": r,
        "principal_vector": principal,
        "selected_index": selected_index,
        "selected_epsilon": epsilons[selected_index],
    }


results = {
    name: spectral_scale_selection(
        points, N_EPSILONS, r=R_WEIGHT
    )
    for name, points in point_clouds.items()
}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for column, (name, result) in enumerate(results.items()):
    eps = result["epsilons"]
    selected = result["selected_epsilon"]

    ax = axes[0, column]
    ax.plot(eps, result["functional"], color="#277da1")
    ax.axvline(selected, color="#d1495b", linestyle="--", label=fr"$\epsilon_{{crit}}={selected:.4f}$")
    ax.set(
        title=name, xlabel=r"$\epsilon$",
        ylabel=r"$r\log|C|+\log|E|$"
    )
    ax.legend()

    ax = axes[1, column]
    ax.plot(eps, result["principal_vector"], color="#f8961e")
    ax.axvline(selected, color="#d1495b", linestyle="--")
    ax.set(xlabel=r"$\epsilon$", ylabel=r"principal coordinate $v_i^*$")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, (name, points) in zip(axes, point_clouds.items()):
    result = results[name]
    epsilon = result["selected_epsilon"]
    D = squareform(pdist(points))
    edge_i, edge_j = np.where(np.triu((D <= 2.0 * epsilon) & (D > 0), k=1))
    for i, j in zip(edge_i, edge_j):
        ax.plot(points[[i, j], 0], points[[i, j], 1], color="#90a4ae", linewidth=0.35, alpha=0.45)
    ax.scatter(points[:, 0], points[:, 1], s=15, color="#1d3557", edgecolor="none", zorder=2)
    idx = result["selected_index"]
    ax.set(
        title=(fr"{name} at $\epsilon_{{crit}}={epsilon:.4f}$" + "\n"
               f"|C|={result['components'][idx]}, |E|={result['edges'][idx]}"),
        xlim=(0, 1), ylim=(0, 1), xlabel="x", ylabel="y"
    )
    ax.set_aspect("equal")
plt.show()

## Numerical summary

The Matérn-II hard-core distance is a simulation parameter, while $\epsilon_{\mathrm{crit}}$ is selected from the global filtration profile; they need not coincide. Because graph edges appear at distance $2\epsilon$, the first possible Matérn-II edge should occur near $h/2$. The summary reports both quantities for orientation.

In [ ]:
header = f"{'process':24s} {'n':>4s} {'diameter':>10s} {'epsilon_crit':>13s} {'|C|':>5s} {'|E|':>7s}"
print(header)
print("-" * len(header))
for name, points in point_clouds.items():
    result = results[name]
    idx = result["selected_index"]
    print(
        f"{name:24s} {len(points):4d} {result['diameter']:10.4f} "
        f"{result['selected_epsilon']:13.4f} {result['components'][idx]:5d} {result['edges'][idx]:7d}"
    )

matern_min_distance = pdist(matern2_points).min()
print(f"\nConfigured Matérn-II hard-core radius R   = {HARDCORE_RADIUS:.4f}")
print(f"Configured minimum separation 2R       = {HARDCORE_DISTANCE:.4f}")
print(f"Observed minimum retained distance       = {matern_min_distance:.4f}")
print(f"First-edge epsilon (minimum distance/2) = {matern_min_distance / 2:.4f}")

# Requested strict-grid checks.
for name, result in results.items():
    eps = result["epsilons"]
    assert 0 < eps[0] and np.all(np.diff(eps) > 0) and eps[-1] < result["diameter"] / 2
assert matern_min_distance + 1e-12 >= HARDCORE_DISTANCE
print("All strict-grid and hard-core checks passed.")

## Parameter study over $r$, grid fineness $N$, and hard-core distance $R$

We use 100 independent realizations for the PPP and for each of three Matérn-II hard-core radii $R$. Following the thesis convention, retained points have separation at least $2R$. Every cloud is evaluated for $r\in\{0.1,0.25,0.5,1,2,4,10\}$ and $N\in\{25,50,100,200,400\}$. Pair distances and single-linkage merge distances are computed once per cloud and reused across the full parameter grid.

In [ ]:
R_GRID = np.array([0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0])
N_GRID = np.array([25, 50, 100, 200, 400])
HARDCORE_RADII = np.array([0.015, 0.025, 0.04])
N_REALIZATIONS = 100

PROCESS_SPECS = [("PPP", None)] + [
    (fr"Matérn II, $R={radius:.3f}$", float(radius))
    for radius in HARDCORE_RADII
]


def prepare_geometry(points):
    distances = pdist(points)
    return {
        "n": len(points),
        "diameter": distances.max(),
        "pair_distances": np.sort(distances),
        "merge_distances": linkage(distances, method="single")[:, 2],
    }


def counts_from_geometry(geometry, n_epsilons):
    epsilons = np.linspace(
        0.0, 0.5 * geometry["diameter"], n_epsilons + 2
    )[1:-1]
    thresholds = 2.0 * epsilons
    edges = np.searchsorted(geometry["pair_distances"], thresholds, side="right")
    merges = np.searchsorted(geometry["merge_distances"], thresholds, side="right")
    components = geometry["n"] - merges
    return epsilons, components, edges


def select_from_counts(epsilons, components, edges, r):
    functional = r * np.log(components) + np.log(np.maximum(edges, 1))
    principal = principal_meet_vector(functional)
    maximizers = np.flatnonzero(np.isclose(
        principal, principal.max(), rtol=1e-9, atol=1e-12
    ))
    index = int(maximizers[0])
    return float(epsilons[index]), index


In [ ]:
parameter_rng = np.random.default_rng(SEED + 1)
critical_scales = {
    label: np.empty((len(R_GRID), len(N_GRID), N_REALIZATIONS))
    for label, _ in PROCESS_SPECS
}
selected_indices = {
    label: np.empty((len(R_GRID), len(N_GRID), N_REALIZATIONS), dtype=int)
    for label, _ in PROCESS_SPECS
}

for label, radius in PROCESS_SPECS:
    for realization in range(N_REALIZATIONS):
        if radius is None:
            points = simulate_ppp(PPP_INTENSITY, WINDOW, parameter_rng)
        else:
            points, _ = simulate_matern_type_ii(
                PPP_INTENSITY, 2.0 * radius, WINDOW, parameter_rng
            )
        geometry = prepare_geometry(points)
        for n_index, n_epsilons in enumerate(N_GRID):
            epsilons, components, edges = counts_from_geometry(geometry, n_epsilons)
            for r_index, r in enumerate(R_GRID):
                scale, index = select_from_counts(epsilons, components, edges, r)
                critical_scales[label][r_index, n_index, realization] = scale
                selected_indices[label][r_index, n_index, realization] = index
    print(f"Completed {label}: {N_REALIZATIONS * len(R_GRID) * len(N_GRID):,} selections")

total_selections = len(PROCESS_SPECS) * N_REALIZATIONS * len(R_GRID) * len(N_GRID)
print(f"Total spectral selections: {total_selections:,}")

In [ ]:
# One histogram figure per process: rows are r-values and columns are grid sizes N.
for process_index, (label, _) in enumerate(PROCESS_SPECS):
    process_values = critical_scales[label]
    common_bins = np.linspace(process_values.min(), process_values.max(), 21)
    color = "#4c78a8" if process_index == 0 else "#f28e2b"
    fig, axes = plt.subplots(
        len(R_GRID), len(N_GRID), figsize=(15, 14),
        sharex=True, sharey=True, constrained_layout=True
    )
    for r_index, r in enumerate(R_GRID):
        for n_index, n_epsilons in enumerate(N_GRID):
            ax = axes[r_index, n_index]
            values = process_values[r_index, n_index]
            ax.hist(values, bins=common_bins, color=color, alpha=0.85, edgecolor="white")
            if r_index == 0:
                ax.set_title(fr"$N={n_epsilons}$")
            if n_index == 0:
                ax.set_ylabel(fr"$r={r:g}$\nfrequency")
            if r_index == len(R_GRID) - 1:
                ax.set_xlabel(r"$\epsilon_{\mathrm{crit}}$")
    fig.suptitle(label + " — chosen-scale histograms", fontsize=15)
    plt.show()

for label, _ in PROCESS_SPECS:
    assert critical_scales[label].shape == (len(R_GRID), len(N_GRID), N_REALIZATIONS)
    assert np.all(np.isfinite(critical_scales[label]))


## Mean and median chosen scales

Each table has functional weight $r$ in its rows and epsilon-grid size $N$ in its columns.

In [ ]:
def print_summary_table(title, matrix):
    header = "r \\ N  " + "".join(f"{n:>11d}" for n in N_GRID)
    print(title)
    print(header)
    print("-" * len(header))
    for r, row in zip(R_GRID, matrix):
        print(f"{r:>6g}  " + "".join(f"{value:11.6f}" for value in row))


for label, _ in PROCESS_SPECS:
    values = critical_scales[label]
    print("\n" + "=" * 72)
    print(label)
    print("=" * 72)
    print_summary_table("Mean epsilon_crit", values.mean(axis=2))
    print()
    print_summary_table("Median epsilon_crit", np.median(values, axis=2))
